In [8]:
from IPython.core import display_functions
from IPython.core import display_functions
from IPython.core import display_functions
import re
import json
from langchain_core.documents import Document
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from sentence_transformers import CrossEncoder

# 1. Re-load ChromaDB Dense Vectorstore
print("Loading ChromaDB Dense Index...")
embeddings = OllamaEmbeddings(model="nomic-embed-text")
vectorstore = Chroma(
    persist_directory="data/chroma_db",             
    embedding_function=embeddings,
    collection_name="code_lens_collection"
)

# 2. Re-load Documents to build the BM25 Sparse Index
print("Building BM25 Sparse Index...")
with open("../data/chunks.json", "r") as f:
    chunk_data = json.load(f)
documents = [Document(page_content=item["page_content"], metadata=item["metadata"]) for item in chunk_data]

bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 20 

dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 20})

# 3. Load the Cross-Encoder for precision Reranking
print("Loading Cross-Encoder model (ms-marco-MiniLM-L-6-v2)...")
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print("All systems loaded!")


Loading ChromaDB Dense Index...
Building BM25 Sparse Index...
Loading Cross-Encoder model (ms-marco-MiniLM-L-6-v2)...


c:\Users\acer\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\acer\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2005.50it/s]


All systems loaded!


Hybrid Search & Reranking Logic

In [9]:
# --- CELL 2 ---
def reciprocal_rank_fusion(results_list, k=60):
    """Fuses multiple ranked lists using Reciprocal Rank Fusion (RRF)."""
    fused_scores = {}
    for docs in results_list:
        for rank, doc in enumerate(docs):
            doc_str = doc.page_content
            if doc_str not in fused_scores:
                fused_scores[doc_str] = {"doc": doc, "score": 0}
            fused_scores[doc_str]["score"] += 1 / (rank + k)
    
    # Sort by fused score descending
    reranked_results = [
        item["doc"] for item in sorted(fused_scores.values(), key=lambda x: x["score"], reverse=True)
    ]
    return reranked_results

def get_hybrid_results(query: str):
    """Gets top 20 from BM25, top 20 from Dense, fuses them, and returns top 20."""
    dense_docs = dense_retriever.invoke(query)
    sparse_docs = bm25_retriever.invoke(query)
    
    fused_docs = reciprocal_rank_fusion([dense_docs, sparse_docs])
    return fused_docs[:20]

def rerank_documents(query: str, documents: list, top_k: int = 5):
    """Scores documents tightly against the query using a Cross-Encoder."""
    if not documents:
        return []
        
    # Prepare pairs for the CrossEncoder: [(query, doc_text), (query, doc_text)...]
    pairs = [[query, doc.page_content] for doc in documents]
    scores = cross_encoder.predict(pairs)
    
    # Attach scores to documents and sort
    scored_docs = list(zip(documents, scores))
    scored_docs.sort(key=lambda x: x[1], reverse=True)
    
    return [doc for doc, score in scored_docs[:top_k]]


 The Intent Router

In [11]:
# --- CELL 3 ---
def route_query(query: str):
    """
    Scans the query for file paths. 
    If a path is found, does a filtered search.
    Otherwise, runs the full hybrid + rerank pipeline.
    """
    # Simple regex looking for typical file extensions (e.g., index.js)
    file_path_match = re.search(r'([a-zA-Z0-9_\-\./]+\.(?:py|js|ts|jsx|tsx|md))', query)
    
    if file_path_match:
        target_file = file_path_match.group(1)
        print(f"--> [Intent Router]: Detected targeted file query for '{target_file}'")
        
        # We can't do a perfect "$contains" filter easily in base Chroma depending on the version,
        # so we fetch 20 and filter in python for safety during the prototype
        candidate_docs = dense_retriever.invoke(query)
        filtered_docs = [d for d in candidate_docs if target_file in d.metadata.get('source', '')]
        
        return filtered_docs[:5] if filtered_docs else candidate_docs[:5]
        
    else:
        print("--> [Intent Router]: Detected global query. Running Hybrid Search + Reranking...")
        candidate_docs = get_hybrid_results(query)
        final_docs = rerank_documents(query, candidate_docs, top_k=5)
        return final_docs


Test the Brain!

In [12]:
# --- CELL 4 ---
print("=== TEST 1: Global Semantic Query ===")
# Should trigger Hybrid + Reranker
results1 = route_query("How are tasks marked as complete?")
for i, d in enumerate(results1):
    print(f"\n{i+1}. [{d.metadata.get('source', 'Unknown').split('/')[-1]}]")
    print(f"{d.page_content[:150]}...")

print("\n\n=== TEST 2: Targeted File Query ===")
# Should trigger the Regex File Router
results2 = route_query("What does the index.js file do?")
for i, d in enumerate(results2):
    print(f"\n{i+1}. [{d.metadata.get('source', 'Unknown').split('/')[-1]}]")
    print(f"{d.page_content[:150]}...")


=== TEST 1: Global Semantic Query ===
--> [Intent Router]: Detected global query. Running Hybrid Search + Reranking...

1. [C:\Users\acer\AppData\Local\Temp\tmpec627czq\index.js]
// Arrays to keep track of each task's state
const taskTitles = [];
const taskComplete = [];

// Create a new task by adding to the arrays
// A new ta...

2. [C:\Users\acer\AppData\Local\Temp\tmpec627czq\index.js]
function logTaskState(taskIndex) {
  const title = taskTitles[taskIndex];
  const complete = taskComplete[taskIndex];
  console.log(`${title} has${com...

3. [C:\Users\acer\AppData\Local\Temp\tmpec627czq\index.js]
function completeTask(taskIndex) {
  taskComplete[taskIndex] = true;
}...

4. [C:\Users\acer\AppData\Local\Temp\tmpec627czq\index.js]
// DRIVER CODE BELOW

newTask("Clean Cat Litter"); // task 0
newTask("Do Laundry"); // task 1

logTaskState(0); // Clean Cat Litter has not been compl...

5. [C:\Users\acer\AppData\Local\Temp\tmpec627czq\index.js]
function newTask(title) {
  taskTitles.push(t